# Unsupervised Learning Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: K-Means from scratch

In [ ]:
```python

import math

import random

def euclidean_distance(a, b):

    return math.sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))

def kmeans(data, k, max_iterations=100, seed=42):

    random.seed(seed)

    n_features = len(data[0])

    centroids = random.sample(data, k)

    for iteration in range(max_iterations):

        clusters = [[] for _ in range(k)]

        assignments = []

        for point in data:

            distances = [euclidean_distance(point, c) for c in centroids]

            nearest = distances.index(min(distances))

            clusters[nearest].append(point)

            assignments.append(nearest)

        new_centroids = []

        for cluster in clusters:

            if len(cluster) == 0:

                new_centroids.append(random.choice(data))

                continue

            centroid = [

                sum(point[j] for point in cluster) / len(cluster)

                for j in range(n_features)

            ]

            new_centroids.append(centroid)

        if all(

            euclidean_distance(old, new) < 1e-6

            for old, new in zip(centroids, new_centroids)

        ):

            print(f"  Converged at iteration {iteration + 1}")

            break

        centroids = new_centroids

    return assignments, centroids

In [ ]:
```

### Step 2: Elbow method and silhouette score

In [ ]:
```python

def compute_inertia(data, assignments, centroids):

    total = 0.0

    for point, cluster_id in zip(data, assignments):

        total += euclidean_distance(point, centroids[cluster_id]) ** 2

    return total

def silhouette_score(data, assignments):

    n = len(data)

    if n < 2:

        return 0.0

    clusters = {}

    for i, c in enumerate(assignments):

        clusters.setdefault(c, []).append(i)

    if len(clusters) < 2:

        return 0.0

    scores = []

    for i in range(n):

        own_cluster = assignments[i]

        own_members = [j for j in clusters[own_cluster] if j != i]

        if len(own_members) == 0:

            scores.append(0.0)

            continue

        a = sum(euclidean_distance(data[i], data[j]) for j in own_members) / len(own_members)

        b = float("inf")

        for cluster_id, members in clusters.items():

            if cluster_id == own_cluster:

                continue

            avg_dist = sum(euclidean_distance(data[i], data[j]) for j in members) / len(members)

            b = min(b, avg_dist)

        if max(a, b) == 0:

            scores.append(0.0)

        else:

            scores.append((b - a) / max(a, b))

    return sum(scores) / len(scores)

def find_best_k(data, max_k=10):

    print("Elbow method:")

    inertias = []

    for k in range(1, max_k + 1):

        assignments, centroids = kmeans(data, k)

        inertia = compute_inertia(data, assignments, centroids)

        inertias.append(inertia)

        print(f"  K={k}: inertia={inertia:.2f}")

    print("\nSilhouette scores:")

    for k in range(2, max_k + 1):

        assignments, centroids = kmeans(data, k)

        score = silhouette_score(data, assignments)

        print(f"  K={k}: silhouette={score:.4f}")

    return inertias

In [ ]:
```

### Step 3: DBSCAN from scratch

In [ ]:
```python

def dbscan(data, eps, min_samples):

    n = len(data)

    labels = [-1] * n

    cluster_id = 0

    def region_query(point_idx):

        neighbors = []

        for i in range(n):

            if euclidean_distance(data[point_idx], data[i]) <= eps:

                neighbors.append(i)

        return neighbors

    visited = [False] * n

    for i in range(n):

        if visited[i]:

            continue

        visited[i] = True

        neighbors = region_query(i)

        if len(neighbors) < min_samples:

            labels[i] = -1

            continue

        labels[i] = cluster_id

        seed_set = list(neighbors)

        seed_set.remove(i)

        j = 0

        while j < len(seed_set):

            q = seed_set[j]

            if not visited[q]:

                visited[q] = True

                q_neighbors = region_query(q)

                if len(q_neighbors) >= min_samples:

                    for nb in q_neighbors:

                        if nb not in seed_set:

                            seed_set.append(nb)

            if labels[q] == -1:

                labels[q] = cluster_id

            j += 1

        cluster_id += 1

    return labels

In [ ]:
```

### Step 4: Gaussian Mixture Model (EM algorithm)

In [ ]:
```python

def gmm(data, k, max_iterations=100, seed=42):

    random.seed(seed)

    n = len(data)

    d = len(data[0])

    indices = random.sample(range(n), k)

    means = [list(data[i]) for i in indices]

    variances = [1.0] * k

    weights = [1.0 / k] * k

    def gaussian_pdf(x, mean, variance):

        d = len(x)

        coeff = 1.0 / ((2 * math.pi * variance) ** (d / 2))

        exponent = -sum((xi - mi) ** 2 for xi, mi in zip(x, mean)) / (2 * variance)

        return coeff * math.exp(max(exponent, -500))

    for iteration in range(max_iterations):

        responsibilities = []

        for i in range(n):

            probs = []

            for j in range(k):

                probs.append(weights[j] * gaussian_pdf(data[i], means[j], variances[j]))

            total = sum(probs)

            if total == 0:

                total = 1e-300

            responsibilities.append([p / total for p in probs])

        old_means = [list(m) for m in means]

        for j in range(k):

            r_sum = sum(responsibilities[i][j] for i in range(n))

            if r_sum < 1e-10:

                continue

            weights[j] = r_sum / n

            for dim in range(d):

                means[j][dim] = sum(

                    responsibilities[i][j] * data[i][dim] for i in range(n)

                ) / r_sum

            variances[j] = sum(

                responsibilities[i][j]

                * sum((data[i][dim] - means[j][dim]) ** 2 for dim in range(d))

                for i in range(n)

            ) / (r_sum * d)

            variances[j] = max(variances[j], 1e-6)

        shift = sum(

            euclidean_distance(old_means[j], means[j]) for j in range(k)

        )

        if shift < 1e-6:

            print(f"  GMM converged at iteration {iteration + 1}")

            break

    assignments = []

    for i in range(n):

        assignments.append(responsibilities[i].index(max(responsibilities[i])))

    return assignments, means, weights, responsibilities

In [ ]:
```

### Step 5: Generate test data and run everything

In [ ]:
```python

def make_blobs(centers, n_per_cluster=50, spread=0.5, seed=42):

    random.seed(seed)

    data = []

    true_labels = []

    for label, (cx, cy) in enumerate(centers):

        for _ in range(n_per_cluster):

            x = cx + random.gauss(0, spread)

            y = cy + random.gauss(0, spread)

            data.append([x, y])

            true_labels.append(label)

    return data, true_labels

def make_moons(n_samples=200, noise=0.1, seed=42):

    random.seed(seed)

    data = []

    labels = []

    n_half = n_samples // 2

    for i in range(n_half):

        angle = math.pi * i / n_half

        x = math.cos(angle) + random.gauss(0, noise)

        y = math.sin(angle) + random.gauss(0, noise)

        data.append([x, y])

        labels.append(0)

    for i in range(n_half):

        angle = math.pi * i / n_half

        x = 1 - math.cos(angle) + random.gauss(0, noise)

        y = 1 - math.sin(angle) - 0.5 + random.gauss(0, noise)

        data.append([x, y])

        labels.append(1)

    return data, labels

if __name__ == "__main__":

    centers = [[2, 2], [8, 3], [5, 8]]

    data, true_labels = make_blobs(centers, n_per_cluster=50, spread=0.8)

    print("=== K-Means on 3 blobs ===")

    assignments, centroids = kmeans(data, k=3)

    print(f"  Centroids: {[[round(c, 2) for c in cent] for cent in centroids]}")

    sil = silhouette_score(data, assignments)

    print(f"  Silhouette score: {sil:.4f}")

    print("\n=== Elbow Method ===")

    find_best_k(data, max_k=6)

    print("\n=== DBSCAN on 3 blobs ===")

    db_labels = dbscan(data, eps=1.5, min_samples=5)

    n_clusters = len(set(db_labels) - {-1})

    n_noise = db_labels.count(-1)

    print(f"  Found {n_clusters} clusters, {n_noise} noise points")

    print("\n=== GMM on 3 blobs ===")

    gmm_assignments, gmm_means, gmm_weights, _ = gmm(data, k=3)

    print(f"  Means: {[[round(m, 2) for m in mean] for mean in gmm_means]}")

    print(f"  Weights: {[round(w, 3) for w in gmm_weights]}")

    gmm_sil = silhouette_score(data, gmm_assignments)

    print(f"  Silhouette score: {gmm_sil:.4f}")

    print("\n=== DBSCAN on moons (non-spherical clusters) ===")

    moon_data, moon_labels = make_moons(n_samples=200, noise=0.1)

    moon_db = dbscan(moon_data, eps=0.3, min_samples=5)

    n_moon_clusters = len(set(moon_db) - {-1})

    n_moon_noise = moon_db.count(-1)

    print(f"  Found {n_moon_clusters} clusters, {n_moon_noise} noise points")

    print("\n=== K-Means on moons (will fail to separate) ===")

    moon_km, moon_centroids = kmeans(moon_data, k=2)

    moon_sil = silhouette_score(moon_data, moon_km)

    print(f"  Silhouette score: {moon_sil:.4f}")

    print("  K-Means splits moons poorly because they are not spherical")

    print("\n=== Anomaly detection with DBSCAN ===")

    anomaly_data = list(data)

    anomaly_data.append([20.0, 20.0])

    anomaly_data.append([-5.0, -5.0])

    anomaly_data.append([15.0, 0.0])

    anomaly_labels = dbscan(anomaly_data, eps=1.5, min_samples=5)

    anomalies = [

        anomaly_data[i]

        for i in range(len(anomaly_labels))

        if anomaly_labels[i] == -1

    ]

    print(f"  Detected {len(anomalies)} anomalies")

    for a in anomalies[-3:]:

        print(f"    Point {[round(v, 2) for v in a]}")

In [ ]:
```

## Exercises

In [ ]:
1. Implement K-Means++ initialization: instead of picking random centroids, pick the first randomly and each subsequent centroid with probability proportional to its squared distance from the nearest existing centroid. Compare convergence speed to random initialization.
2. Add hierarchical agglomerative clustering to the code. Implement Ward's linkage and produce a dendrogram (as a nested list of merges). Cut it at different levels and compare to K-Means results.
3. Build a simple anomaly detection pipeline: run DBSCAN and GMM on the same data, flag points that both methods agree are outliers (noise in DBSCAN, low probability in GMM). Measure the overlap and discuss when the methods disagree.